In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/spaceship-titanic/sample_submission.csv
/kaggle/input/competitions/spaceship-titanic/train.csv
/kaggle/input/competitions/spaceship-titanic/test.csv


In [2]:
import numpy as np
import pandas as pd
import ydf

# Load data
train_df = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/test.csv")

print("train columns:", train_df.columns.tolist())
print("test columns:", test_df.columns.tolist())

# Fill missing values
for df in [train_df, test_df]:
    if "Age" in df.columns:
        df["Age"] = df["Age"].fillna(df["Age"].median())

    for col in ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())

    for col in ["HomePlanet", "CryoSleep", "Cabin", "Destination", "VIP", "Name"]:
        if col in df.columns:
            mode_val = df[col].mode(dropna=True)
            if len(mode_val) > 0:
                df[col] = df[col].fillna(mode_val.iloc[0])

# Features for model
features = [
    "HomePlanet",
    "CryoSleep",
    "Cabin",
    "Destination",
    "Age",
    "VIP",
    "RoomService",
    "FoodCourt",
    "ShoppingMall",
    "Spa",
    "VRDeck",
    "Name",
]

features = [c for c in features if c in train_df.columns and c in test_df.columns]

print("Using features:", features)

# Build train/test data
train_data = train_df[features + ["Transported"]].copy()
test_data = test_df[features].copy()

# Train model
model = ydf.RandomForestLearner(label="Transported").train(train_data)

# Predict
predictions = model.predict(test_data)

# Convert to boolean
if predictions.ndim > 1:
    transported_pred = np.argmax(predictions, axis=1).astype(bool)
else:
    transported_pred = (predictions > 0.5)

# Submission
submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Transported": transported_pred
})

submission.to_csv("/kaggle/working/submission.csv", index=False)

print(submission.head())
print("Saved: /kaggle/working/submission.csv")

train columns: ['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Name', 'Transported']
test columns: ['PassengerId', 'HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Name']
Using features: ['HomePlanet', 'CryoSleep', 'Cabin', 'Destination', 'Age', 'VIP', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck', 'Name']
Feature Name is a CATEGORICAL feature whose dictionary has a single element. The feature will not be useful during model training.
Train model on 8693 examples


/tmp/ipykernel_17/1407210122.py:25: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(mode_val.iloc[0])


Model trained in 0:00:02.087221
  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01         True
Saved: /kaggle/working/submission.csv


In [3]:
import numpy as np
import pandas as pd
import ydf

train_df = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/train.csv")
test_df = pd.read_csv("/kaggle/input/competitions/spaceship-titanic/test.csv")

def preprocess(df):
    df = df.copy()

    # Split Cabin into deck / num / side
    if "Cabin" in df.columns:
        cabin_split = df["Cabin"].fillna("Unknown/0/U").str.split("/", expand=True)
        df["CabinDeck"] = cabin_split[0]
        df["CabinNum"] = pd.to_numeric(cabin_split[1], errors="coerce")
        df["CabinSide"] = cabin_split[2]

    # Total spend
    spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
    for col in spend_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    df["TotalSpend"] = df[spend_cols].sum(axis=1)

    # Fill numeric
    for col in ["Age", "CabinNum", "TotalSpend"] + spend_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())

    # Fill categorical
    for col in ["HomePlanet", "CryoSleep", "Destination", "VIP", "CabinDeck", "CabinSide"]:
        if col in df.columns:
            mode_val = df[col].mode(dropna=True)
            if len(mode_val) > 0:
                df[col] = df[col].fillna(mode_val.iloc[0])

    return df

train_df = preprocess(train_df)
test_df = preprocess(test_df)

features = [
    "HomePlanet", "CryoSleep", "Destination", "Age", "VIP",
    "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck",
    "CabinDeck", "CabinNum", "CabinSide", "TotalSpend"
]

train_data = train_df[features + ["Transported"]]
test_data = test_df[features]

model = ydf.GradientBoostedTreesLearner(label="Transported").train(train_data)
predictions = model.predict(test_data)

if predictions.ndim > 1:
    transported_pred = np.argmax(predictions, axis=1).astype(bool)
else:
    transported_pred = (predictions > 0.5)

submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Transported": transported_pred
})

submission.to_csv("/kaggle/working/submission.csv", index=False)
print(submission.head())

Train model on 8693 examples


/tmp/ipykernel_17/2669169691.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(mode_val.iloc[0])
/tmp/ipykernel_17/2669169691.py:36: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].fillna(mode_val.iloc[0])


Model trained in 0:00:01.320138
  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01         True
